# The Artificial Neuron
## Turn several measurements into one useful response

Imagine a small greenhouse controller that receives two measurements: how dry the soil is and how warm the air is. Dry soil should encourage watering. Warm air might add some encouragement, but not as much. How can we combine these signals into one adjustable score?

We could write separate rules for every possible pair of readings. A simpler starting point is to assign each input an influence, add those contributions, and decide how to respond. That small computation is the building block we will unpack here.

An artificial neuron is one adjustable computation that can help construct useful representations of data. By the end, you should be able to calculate a neuron's output, explain what each parameter does, reproduce it with PyTorch, and describe a limitation of a single threshold neuron.

The greenhouse is an invented teaching example. Its numbers are hand-chosen to make the arithmetic clear, not validated instructions for growing plants. Open the [code companion](02_The_Artificial_Neuron.ipynb) after reading the lesson.

## Give each input a different influence

Let $x_1$ be a soil-dryness reading between 0 and 1: 0 means very wet and 1 means very dry. Let $x_2$ be a warmth reading, also between 0 and 1: 0 means cool and 1 means warm. These are already normalized readings supplied by our fictional sensors; we are not estimating a scaling rule from a dataset.

Suppose the readings are $x_1=0.8$ and $x_2=0.4$. We want dryness to have twice the influence of warmth for the same numerical change, so choose **weights** $w_1=2$ and $w_2=1$. A weight is a multiplier attached to an input.

| Input | Reading | Weight | Contribution |
| --- | ---: | ---: | ---: |
| Dryness | 0.8 | 2 | $2\times0.8=1.6$ |
| Warmth | 0.4 | 1 | $1\times0.4=0.4$ |
| Total | | | 2.0 |

A **weighted sum** adds those contributions. It allows one input to count more strongly than another. Holding everything else fixed, increasing dryness by 0.1 increases this sum by 0.2; increasing warmth by 0.1 increases it by 0.1.

A negative weight would reverse that direction. If an input measured wetness rather than dryness, a negative weight might make sense. A zero weight makes that input have no direct effect on this neuron's score. These statements describe the calculation, not causal effects in the real world.

Weight sizes are meaningful only alongside feature units and ranges. A weight on degrees Celsius cannot be compared directly with a weight on a fraction merely by looking at their magnitudes.

## Why add a bias?

The weighted sum is 2.0, but what should count as enough encouragement? We need a reference point. Suppose our chosen cutoff for the sum is 1.5. Subtracting that cutoff gives a score centered on zero:

$$z=w_1x_1+w_2x_2+b=2(0.8)+1(0.4)-1.5=0.5.$$

Here $z$ is the score before the final response rule, and $b=-1.5$ is the **bias**: an adjustable offset independent of the current input. The symbols $x_1,x_2$ are the two readings and $w_1,w_2$ are their weights.

The bias lets the unit set its baseline response. Without it, the score would be zero whenever both inputs are zero, and a zero-score boundary would have to pass through the origin. With it, we can move that boundary.

A bias is added once per example, not once per input. It is not an error term or a statement about social bias. In this equation it simply means an offset, just like the intercept in regression.

For any number of inputs, the same calculation is

$$z=\sum_{i=1}^{d}w_i x_i+b.$$

$d$ is the number of input features, $i$ identifies one feature, $x_i$ is its value, and $w_i$ is its weight. The summation symbol means “multiply each matching input and weight, then add all $d$ products.” Finally add $b$. A unit with $d$ inputs has $d+1$ adjustable parameters when it includes a bias.

The weights and bias are **parameters**: values the model stores and can learn during training. The inputs change from example to example; the same parameters are reused across examples. We hand-set them here to isolate the computation.

## The activation decides what to pass onward

The score alone is not necessarily the output we want. An **activation function** is a rule applied to that score to produce the unit's output:

$$a=\phi(z).$$

$a$ is the output, and $\phi$ (read “phi”) is the chosen activation function. The full unit—weighted sum, bias, and activation—is commonly called an **artificial neuron**. It is a simplified computational building block, not a realistic model of a biological nerve cell.

```text
Dryness x1 ── multiply by w1 ──┐
                              ├── add contributions and b ── z ── activation φ ── a
Warmth  x2 ── multiply by w2 ──┘
```

For our score $z=0.5$, different activations give different meanings:

| Activation | Rule | Output at $z=0.5$ | Interpretation |
| --- | --- | ---: | --- |
| Identity | $\phi(z)=z$ | 0.5 | Keep the signed score. |
| Hard step | 1 if $z>0$, otherwise 0 | 1 | Make a yes/no decision. |
| ReLU | $\phi(z)=\max(0,z)$ | 0.5 | Keep positive values; replace negative values by zero. |
| Sigmoid | $\phi(z)=1/(1+e^{-z})$ | about 0.622 | Map the score into the interval between 0 and 1. |

$\max(0,z)$ means choose the larger of zero and $z$. ReLU stands for rectified linear unit. In sigmoid, $e\approx2.718$ is the base of the exponential. Substituting $z=0.5$ gives $e^{-0.5}\approx0.607$, so the output is $1/(1+0.607)\approx0.622$.

A value between 0 and 1 is not automatically a trustworthy probability. With appropriate binary classification training, sigmoid output can be interpreted as a probability estimate. Our hand-set greenhouse value is only an illustrative response score, not a measured 62.2% chance of anything.

The hard step is often associated with the classical **perceptron**, a threshold-based neuron. This discontinuous rule is unsuitable for ordinary gradient-based training: it is flat away from its jump and has no derivative at the jump. Modern networks often use other activations. The important distinction here is between the pre-activation score $z$ and the output $a$.

## Follow several examples through the same neuron

Keep the weights $(2,1)$ and bias $-1.5$ fixed. Only change the inputs:

| Dryness $x_1$ | Warmth $x_2$ | $2x_1+x_2-1.5$ | Step output | Sigmoid output |
| ---: | ---: | ---: | ---: | ---: |
| 0.8 | 0.4 | 0.5 | 1 | 0.622 |
| 0.2 | 0.4 | -0.7 | 0 | 0.332 |
| 0.5 | 0.5 | 0.0 | 0 | 0.500 |
| 0.9 | 0.9 | 1.2 | 1 | 0.769 |

For the second row, the contributions are $2(0.2)=0.4$ and $1(0.4)=0.4$. Adding the bias gives $0.4+0.4-1.5=-0.7$. The step returns 0, but sigmoid retains a graded response of about 0.332.

We deliberately define the step with a strict greater-than comparison. At exactly zero, it returns 0. An alternative convention could return 1 at zero; either is valid if used consistently. Sigmoid gives exactly 0.5 there. In this lesson, thresholding sigmoid with a strict greater-than 0.5 matches the step rule mathematically.

A **forward pass** means computing outputs from inputs using the current parameters. Nothing in a forward pass requires the parameters to change. **Training** is the separate process that uses data and a loss to update them. The companion only performs forward passes and controlled parameter changes, so it needs no training/validation/test split and makes no accuracy claim about unseen data.

## See weights and bias as geometry

If we make a binary decision by checking whether $z>0$, the dividing boundary is where $z=0$:

$$2x_1+x_2-1.5=0,\qquad x_2=1.5-2x_1.$$

This is a straight line in the two-input plane. The weights determine its orientation, while changing the bias with the weights fixed shifts it parallel to itself.

For example, set $b=-1.0$ instead of $-1.5$. Every score increases by 0.5. The new boundary is $x_2=1.0-2x_1$, so more input pairs now have positive scores. With $b=-2.0$, the boundary is $x_2=2.0-2x_1$, and fewer pairs receive output 1.

If we instead change the dryness weight, we change the line's slope. In general, if $w_2$ is nonzero, rearranging the boundary gives $x_2=-(w_1/w_2)x_1-b/w_2$. If $w_2=0$ and $w_1$ is nonzero, it is a vertical line. If both weights are zero, the output depends only on the bias.

Sigmoid changes the output scale, but thresholding it at 0.5 still gives this same straight boundary. A nonlinear activation does not automatically make one neuron's thresholded decision boundary curved.

## One example or a whole batch: the same arithmetic

A **batch** groups examples so we can compute their outputs together. A **tensor** is an array of numbers; its **shape** gives the sizes of its axes. For the four examples above, PyTorch uses this arrangement:

| Quantity | Shape | Meaning |
| --- | --- | --- |
| Input $X$ | $(4,2)$ | Four examples, two features each |
| Weight $W$ | $(1,2)$ | One output unit, two input weights |
| Bias $b$ | $(1,)$ | One offset for that unit |
| Score $Z$ | $(4,1)$ | One score for each example |
| Activated output $A$ | $(4,1)$ | One response for each example |

In PyTorch's layout, the batch computation is

$$Z=XW^\mathsf{T}+b,\qquad A=\phi(Z).$$

$X$ contains the input rows, $W$ stores the weights, and $W^\mathsf{T}$ is its **transpose**, which exchanges rows and columns. Multiplying a $(4,2)$ matrix by a $(2,1)$ matrix gives a $(4,1)$ result. $b$ is reused for every row, an operation called **broadcasting**. The activation is then applied separately to each score.

`nn.Linear(2, 1)` implements the weighted sums plus bias. Despite the name “Linear,” this is technically an **affine** transformation when a bias is present: a linear transformation followed by an offset. The module does not include sigmoid or ReLU; those are separate operations. Checking these shapes prevents a common source of errors.

## A single threshold neuron has a real limitation

Consider two binary switches. **AND** should output 1 only when both are on. We can implement it with weights $(1,1)$, bias $-1.5$, and the step rule. Scores for $(0,0),(0,1),(1,0),(1,1)$ are $-1.5,-0.5,-0.5,0.5$, giving $0,0,0,1$.

**XOR**, meaning “exclusive or,” should output 1 when exactly one switch is on:

| $x_1$ | $x_2$ | XOR target |
| ---: | ---: | ---: |
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

Could different weights and a bias solve it with the same step rule? To classify $(0,0)$ as 0, we need $b\leq0$. To classify the two positive examples, we need $w_1+b>0$ and $w_2+b>0$. Adding those inequalities gives $w_1+w_2+2b>0$. Therefore the score at $(1,1)$ satisfies

$$w_1+w_2+b>-b\geq0.$$

It must be positive, but XOR requires output 0. This contradiction proves there is no parameter choice that solves all four rows with one affine score and this threshold rule. More training cannot fix that expressiveness limit.

The limitation applies to these raw inputs and this kind of unit. Additional features or multiple units with nonlinear processing can solve XOR. The companion verifies the arithmetic behind the contradiction; trying one unsuccessful weight choice alone would not prove impossibility.

## How neurons are used

An identity-output neuron is the familiar single-output affine regression model. Pair its score with sigmoid and binary classification training, and it gives the familiar logistic regression model. Hidden neurons reuse the same weighted-sum building block to produce intermediate values for other units.

The job of a neuron depends on its inputs, parameters, activation, and position in the network. A hidden output is not necessarily a class probability or an easily named concept. Real networks combine many such computations; generative models use these building blocks inside much larger structures, together with appropriate objectives and data.

Practical mistakes to avoid include adding the bias twice, assuming `nn.Linear` includes an activation, mixing up input features with batch rows, and interpreting a large weight without considering input scale. Another mistake is confusing manually assigning weights with learning them. In the companion we deliberately assign weights rather than learn them.

## Recap and self-check

A neuron multiplies each input by its weight, adds the contributions and one bias, then applies an activation. Weights control input influence; bias shifts the baseline; activation determines what is passed onward. The parameters are shared across examples. One threshold neuron can make only a straight separating boundary in a two-feature plane.

- For $x=(0.6,0.2)$, weights $(2,1)$, and bias $-1.5$, what are the score, step output, and ReLU output?
- What happens to every score if the bias increases by 0.3?
- How many parameters does one neuron with five inputs and a bias have?
- Why does sigmoid alone not solve XOR with one affine score and a 0.5 threshold?

Check your reasoning: the score is $1.2+0.2-1.5=-0.1$, and both step and ReLU return 0. Increasing bias adds 0.3 to every score. Five weights plus a bias give six parameters. Sigmoid's 0.5 boundary is still the zero boundary of an affine score, so the XOR contradiction still applies.